## Case Demanding:

- Product Master Transformations:
Perform the following transformations on the product master data and write the results into a table named publish_product:
Replace NULL values in the Color field with N/A.
Enhance the ProductCategoryName field when it is NULL using the following logic:
If ProductSubCategoryName is in (‘Gloves’, ‘Shorts’, ‘Socks’, ‘Tights’, ‘Vests’), set ProductCategoryName to ‘Clothing’.
If ProductSubCategoryName is in (‘Locks’, ‘Lights’, ‘Headsets’, ‘Helmets’, ‘Pedals’, ‘Pumps’), set ProductCategoryName to ‘Accessories’.
If ProductSubCategoryName contains the word ‘Frames’ or is in (‘Wheels’, ‘Saddles’), set ProductCategoryName to ‘Components’.

- Sales Order Transformations:
Join SalesOrderDetail with SalesOrderHeader on SalesOrderId and apply the following transformations:
Calculate LeadTimeInBusinessDays as the difference between OrderDate and ShipDate, excluding Saturdays and Sundays.
Calculate TotalLineExtendedPrice using the formula: OrderQty * (UnitPrice - UnitPriceDiscount).
Write the results into a table named publish_orders, including:
All fields from SalesOrderDetail.
All fields from SalesOrderHeader except SalesOrderId, and rename Freight to TotalOrderFreight.

#### Environment Setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import datetime

#### Products Tranformations

In [0]:
try:
  # Handling nulls on the columns Color and ProductCategoryName
  df_products_store = spark.table('interviewcaseupstart.sales_db.store_products').filter(F.col('ProductPublished') == False)
  df_products = df_products_store.withColumn('Color',F.when(F.col('Color').isNull(),F.lit('N/A')).otherwise(F.col('Color')))
  df_products = (df_products
                 .withColumn('ProductCategoryName',
                               F.when(
                                  (
                                    F.col('ProductCategoryName').isNull() & 
                                    F.col('ProductSubCategoryName').isin('Gloves', 'Shorts', 'Socks', 'Tights', 'Vests')
                                  ),
                                    F.lit('Clothing'))
                                .when(
                                  (
                                    F.col('ProductCategoryName').isNull() & 
                                    F.col('ProductSubCategoryName').isin('Locks', 'Lights', 'Headsets', 'Helmets', 'Pedals', 'Pumps')
                                  ),
                                    F.lit('Accessories')
                                )
                                .when(
                                  F.col('ProductCategoryName').isNull() & 
                                  (
                                    F.col('ProductSubCategoryName').like('%Frames%') |
                                    F.col('ProductSubCategoryName').isin('Wheels', 'Saddles')
                                  ),
                                    F.lit('Components')
                                ).otherwise(F.col('ProductCategoryName'))
                            ))
  df_products_store = (df_products_store
                       .withColumn('ProductPublished',F.lit(True))
                       .withColumn('LastModificationTimeNew',F.lit(datetime.datetime.now().strftime("%Y-%m-%dT%H:%M:%S")).cast(TimestampType())))
except Exception as e:
  dbutils.notebook.exit(e)

In [0]:
try:
  delta_table_products_gold = DeltaTable.forName(spark, 'interviewcaseupstart.sales_db.publish_product')

  (delta_table_products_gold.alias("T")
    .merge(
      df_products.alias("S"),
      "S.ProductID = T.ProductID")
    .whenMatchedUpdate(set =
      {
        "ProductDesc": "S.ProductDesc",
        "ProductNumber": "S.ProductNumber",
        "MakeFlag": "S.MakeFlag",
        "Color": "S.Color",
        "SafetyStockLevel": "S.SafetyStockLevel",
        "ReorderPoint": "S.ReorderPoint",
        "StandardCost": "S.StandardCost",
        "ListPrice": "S.ListPrice",
        "Size": "S.Size",
        "SizeUnitMeasureCode": "S.SizeUnitMeasureCode",
        "WeightUnitMeasureCode": "S.WeightUnitMeasureCode",
        "Weight": "S.Weight",
        "ProductCategoryName": "S.ProductCategoryName",
        "ProductSubcategoryName": "S.ProductSubcategoryName"
      })
    .whenNotMatchedInsertAll()
    .execute()
  )

  delta_table_products_silver = DeltaTable.forName(spark, 'interviewcaseupstart.sales_db.store_products')
  
  (delta_table_products_silver.alias("T")
    .merge(
      df_products_store.alias("S"),
      "T.ProductID = S.ProductID and T.LastModificationTime = S.LastModificationTime and T.ProductPublished = false")
    .whenMatchedUpdate(set =
      {
        "ProductPublished": "S.ProductPublished",
        "LastModificationTime": "S.LastModificationTimeNew"
      })
    .execute()
  )
except Exception as e:
  dbutils.notebook.exit(e)

#### Sales Order Transformations

In [0]:
try:
  # Getting just files not loaded on the store table
  df_order_detail_store = spark.table('interviewcaseupstart.sales_db.store_sales_order_detail').filter(F.col('OrderDetailPublished') == False)
  df_order_header_store = spark.table('interviewcaseupstart.sales_db.store_sales_order_header').filter(F.col('OrderHeaderPublished') == False)

  #Droping not used columns
  df_order_detail_store_join = df_order_detail_store.drop('LastModificationTime','OrderDetailPublished')
  df_order_header_store_join = df_order_header_store.drop('LastModificationTime','OrderHeaderPublished')

  #Creating LeadTimeInBusinessDays column based on just weekdays
  df_order = df_order_detail_store_join.join(df_order_header_store_join,on='SalesOrderID',how='inner')
  df_order = df_order.withColumn("FullLeadTime", F.sequence(F.col("OrderDate"), F.col("ShipDate"), F.expr("INTERVAL 1 DAY")))
  df_order = df_order.withColumn("WeekdaysOnly", F.expr("FILTER(FullLeadTime, d -> DAYOFWEEK(d) != 1 AND DAYOFWEEK(d) != 7)"))
  df_order = df_order.withColumn("LeadTimeInBusinessDays", F.size(F.col("WeekdaysOnly")))

  #Calculating TotalLineExtendedPrice
  df_order = df_order.withColumn("TotalLineExtendedPrice", F.col("OrderQty") * (F.col("UnitPrice") - F.col("UnitPriceDiscount")))

  #Final order DF
  df_order = df_order.drop('FullLeadTime','WeekdaysOnly').withColumnRenamed('Freight','TotalOrderFreight')

  #DF to update the store orders tables
  df_order_detail_store = (df_order_detail_store
                            .withColumn('OrderDetailPublished',F.lit(True))
                            .withColumn('LastModificationTimeNew',F.lit(datetime.datetime.now().strftime("%Y-%m-%dT%H:%M:%S")).cast(TimestampType())))
  df_order_header_store = (df_order_header_store
                            .withColumn('OrderHeaderPublished',F.lit(True))
                            .withColumn('LastModificationTimeNew',F.lit(datetime.datetime.now().strftime("%Y-%m-%dT%H:%M:%S")).cast(TimestampType())))
except Exception as e:
  dbutils.notebook.exit(e)

In [0]:
try:
  delta_table_orders_gold = DeltaTable.forName(spark, 'interviewcaseupstart.sales_db.publish_orders')

  (delta_table_orders_gold.alias("T")
    .merge(
      df_order.alias("S"),
      "S.SalesOrderID = T.SalesOrderID and S.SalesOrderDetailID = T.SalesOrderDetailID and S.ProductID = T.ProductID and S.CustomerID = T.CustomerID")
    .whenMatchedUpdate(set =
      {
        "OrderQty": "S.OrderQty",
        "UnitPrice": "S.UnitPrice",
        "UnitPriceDiscount": "S.UnitPriceDiscount",
        "OrderDate": "S.OrderDate",
        "ShipDate": "S.ShipDate",
        "OnlineOrderFlag": "S.OnlineOrderFlag",
        "AccountNumber": "S.AccountNumber",
        "SalesPersonID": "S.SalesPersonID",
        "TotalOrderFreight": "S.TotalOrderFreight",
        "LeadTimeInBusinessDays": "S.LeadTimeInBusinessDays",
        "TotalLineExtendedPrice": "S.TotalLineExtendedPrice"
      })
    .whenNotMatchedInsertAll()
    .execute()
  )

  delta_table_order_detail_silver = DeltaTable.forName(spark, 'interviewcaseupstart.sales_db.store_sales_order_detail')
  
  (delta_table_order_detail_silver.alias("T")
    .merge(
      df_order_detail_store.alias("S"),
      "T.SalesOrderID = S.SalesOrderID and T.SalesOrderDetailID = S.SalesOrderDetailID and T.LastModificationTime = S.LastModificationTime and T.OrderDetailPublished = false")
    .whenMatchedUpdate(set =
      {
        "OrderDetailPublished": "S.OrderDetailPublished",
        "LastModificationTime": "S.LastModificationTimeNew"
      })
    .execute()
  )

  delta_table_order_header_silver = DeltaTable.forName(spark, 'interviewcaseupstart.sales_db.store_sales_order_header')
  
  (delta_table_order_header_silver.alias("T")
    .merge(
      df_order_header_store.alias("S"),
      "T.SalesOrderID = S.SalesOrderID and T.LastModificationTime = S.LastModificationTime and T.OrderHeaderPublished = false")
    .whenMatchedUpdate(set =
      {
        "OrderHeaderPublished": "S.OrderHeaderPublished",
        "LastModificationTime": "S.LastModificationTimeNew"
      })
    .execute()
  )
except Exception as e:
  dbutils.notebook.exit(e)